# 04 — Fitur Transformer: Sentence-BERT & CodeBERT

**Tujuan:** mengubah tiap soal menjadi satu vektor dari dua transformer pretrained, **tanpa fine-tuning** (*frozen feature extractor*), untuk menjawab **RQ3** (model bahasa umum vs model domain kode).

**Input:** `data_clean.csv` (kolom `text_light`, dari Notebook 01)

**Output** (baris sejajar dengan `data_clean.csv`, terurut menurut `id`):
- `features/sbert.npy` — N × 384 (MiniLM atau bge-small)
- `features/codebert.npy` — N × 768
- `tables/truncation_stats.csv` — berapa soal yang terpotong oleh batas panjang token
- `features/transformer_meta.json` — catatan model dan pooling untuk Bab III-D

**Keputusan desain**
- Teks masukan = **`text_light`** (angka, simbol, dan huruf besar dipertahankan), bukan `text_clean`.
- **Mean pooling** dari hidden state terakhir dengan mask padding. `[CLS]` model yang belum di-fine-tune lemah sebagai representasi kalimat.
- Tanpa fine-tuning, jadi representasi dan classifier tetap terpisah (matriks 5 × 3 tetap adil).
- Embedding tidak memakai label, jadi seluruh soal boleh diproses sekaligus tanpa kebocoran.

**Menjalankan di Kaggle:** aktifkan **Accelerator: GPU** (T4/P100). Model diunduh dari Hugging Face, jadi **Internet harus ON**. Kalau Internet tidak bisa dinyalakan, tambahkan model lewat *Add Input → Models* dan isi `SBERT_ID` / `CODEBERT_ID` dengan path folder model (lihat sel konfigurasi).

## 1. Import & Konfigurasi

In [1]:
import glob
import json
import os

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer

ON_KAGGLE = os.path.exists('/kaggle/input')
OUT_DIR = '/kaggle/working' if ON_KAGGLE else '.'
FEAT_DIR = os.path.join(OUT_DIR, 'features')
TAB_DIR = os.path.join(OUT_DIR, 'tables')
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

# --- Pilihan model (ubah hanya di sini) ---
SBERT_ID = 'BAAI/bge-small-en-v1.5'   # alternatif: 'BAAI/bge-small-en-v1.5'
SBERT_MAXLEN = 512                                    # isi 512 jika memakai bge-small
CODEBERT_ID = 'microsoft/codebert-base'
CODEBERT_MAXLEN = 512
BATCH = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('PERINGATAN: GPU tidak aktif. Ekstraksi tetap jalan tetapi jauh lebih lambat.')


def find_file(name):
    if os.path.exists(name):
        return name
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else None

Device: cuda


## 2. Load Data

In [2]:
DATA_PATH = find_file('data_clean.csv')
assert DATA_PATH is not None, 'data_clean.csv tidak ditemukan (tambahkan Kaggle Dataset-nya)'
df = pd.read_csv(DATA_PATH)
assert df['id'].is_monotonic_increasing and df['id'].is_unique, 'urutan harus menurut id'

texts = df['text_light'].fillna('').astype(str).tolist()
assert all(t.strip() for t in texts), 'ada text_light kosong'
print('Jumlah soal:', len(texts))
print('Contoh     :', texts[0][:200])

Jumlah soal: 3256
Contoh     : You are given an array of integers nums and an integer target , return indices of the two numbers such that they add up to target . You may assume that each input would have exactly one solution , and


## 3. Fungsi Bantu

`token_stats` mengukur panjang teks dalam token **milik tokenizer model itu sendiri**, tanpa pemotongan, lalu menghitung seberapa banyak soal melewati batas panjang. Angka ini dilaporkan sebagai keterbatasan: bagian akhir soal (biasanya *constraints*) adalah yang terpotong.

In [3]:
def token_stats(tokenizer, texts, maxlen, name):
    lens = np.array([len(ids) for ids in tokenizer(texts, add_special_tokens=True, truncation=False)['input_ids']])
    lost = np.maximum(lens - maxlen, 0).sum() / lens.sum()
    return {
        'model': name,
        'max_len': maxlen,
        'tokens_mean': round(float(lens.mean()), 1),
        'tokens_median': float(np.median(lens)),
        'tokens_p95': float(np.percentile(lens, 95)),
        'tokens_max': int(lens.max()),
        'problems_truncated_n': int((lens > maxlen).sum()),
        'problems_truncated_%': round(100 * float((lens > maxlen).mean()), 2),
        'tokens_lost_%': round(100 * float(lost), 2),
    }


def top_similar(X, idx, k=5):
    Xn = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)
    sims = Xn @ Xn[idx]
    order = np.argsort(-sims)
    return [(df.loc[j, 'title'], round(float(sims[j]), 3)) for j in order if j != idx][:k]

## 4. Sentence-BERT

In [4]:
sbert = SentenceTransformer(SBERT_ID, device=device)
sbert.max_seq_length = SBERT_MAXLEN
print('SBERT dimensi:', sbert.get_sentence_embedding_dimension(), '| max_seq_length:', sbert.max_seq_length)

stats_sbert = token_stats(sbert.tokenizer, texts, SBERT_MAXLEN, SBERT_ID)
print(json.dumps(stats_sbert, indent=2))

# encode() mengurutkan menurut panjang secara internal dan mengembalikan urutan ASLI
sbert_X = sbert.encode(texts, batch_size=BATCH, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
print('SBERT shape:', sbert_X.shape)

del sbert
if device == 'cuda':
    torch.cuda.empty_cache()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (9 outputs left out)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (654 > 512). Running this sequence through the model will result in indexing errors


{
  "model": "BAAI/bge-small-en-v1.5",
  "max_len": 512,
  "tokens_mean": 383.0,
  "tokens_median": 342.0,
  "tokens_p95": 758.25,
  "tokens_max": 1935,
  "problems_truncated_n": 633,
  "problems_truncated_%": 19.44,
  "tokens_lost_%": 9.58
}


Batches:   0%|          | 0/102 [00:00<?, ?it/s]

SBERT shape: (3256, 384)


## 5. CodeBERT

Mean pooling dengan mask padding dibuat manual. Teks diurutkan menurut panjang agar batch efisien, lalu hasilnya dikembalikan ke **urutan asli**.

In [5]:
cb_tok = AutoTokenizer.from_pretrained(CODEBERT_ID)
cb_model = AutoModel.from_pretrained(CODEBERT_ID).to(device).eval()
print('CodeBERT hidden size:', cb_model.config.hidden_size)

stats_cb = token_stats(cb_tok, texts, CODEBERT_MAXLEN, CODEBERT_ID)
print(json.dumps(stats_cb, indent=2))


@torch.no_grad()
def embed_batch(batch_texts):
    enc = cb_tok(batch_texts, padding=True, truncation=True, max_length=CODEBERT_MAXLEN,
                 return_tensors='pt').to(device)
    hidden = cb_model(**enc).last_hidden_state                      # (B, T, H)
    mask = enc['attention_mask'].unsqueeze(-1).to(hidden.dtype)     # (B, T, 1)
    pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return pooled.float().cpu().numpy()


def embed_all(all_texts, batch_size):
    order = np.argsort([len(t) for t in all_texts])                 # pendek -> panjang
    out = np.zeros((len(all_texts), cb_model.config.hidden_size), dtype=np.float32)
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        out[idx] = embed_batch([all_texts[i] for i in idx])
        if (start // batch_size) % 20 == 0:
            print(f'  {start + len(idx)}/{len(order)}')
    return out


codebert_X = embed_all(texts, BATCH)
print('CodeBERT shape:', codebert_X.shape)

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

CodeBERT hidden size: 768


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (643 > 512). Running this sequence through the model will result in indexing errors


{
  "model": "microsoft/codebert-base",
  "max_len": 512,
  "tokens_mean": 352.1,
  "tokens_median": 319.0,
  "tokens_p95": 659.5,
  "tokens_max": 1572,
  "problems_truncated_n": 464,
  "problems_truncated_%": 14.25,
  "tokens_lost_%": 6.09
}
  32/3256
  672/3256
  1312/3256
  1952/3256
  2592/3256
  3232/3256
CodeBERT shape: (3256, 768)


## 6. Sanity Check

1. Bentuk benar dan tidak ada NaN/baris nol.
2. **Pooling konsisten terhadap padding:** embedding satu soal yang diproses sendirian harus sama dengan hasilnya saat berada dalam batch. Kalau berbeda, mask padding bermasalah.
3. Soal paling mirip dengan *Two Sum* harus masuk akal secara topik.

In [6]:
for name, X in [('sbert', sbert_X), ('codebert', codebert_X)]:
    assert X.shape[0] == len(df), f'{name}: jumlah baris salah {X.shape}'
    assert np.isfinite(X).all(), f'{name}: ada NaN/inf'
    n_zero = int((np.linalg.norm(X, axis=1) == 0).sum())
    print(f'{name:9s} shape={X.shape}  baris nol={n_zero}  norm rata-rata={np.linalg.norm(X, axis=1).mean():.3f}')
    assert n_zero == 0

# Uji konsistensi padding: 3 soal, satu per satu vs dalam batch
probe = [0, len(texts) // 2, len(texts) - 1]
alone = np.vstack([embed_batch([texts[i]]) for i in probe])
max_diff = float(np.abs(alone - codebert_X[probe]).max())
print('Selisih maksimum (sendiri vs batch):', round(max_diff, 5))
assert max_diff < 5e-3, 'embedding tidak konsisten terhadap padding'

idx = int(df.index[df['id'] == 1][0]) if (df['id'] == 1).any() else 0
print()
print('Soal acuan:', df.loc[idx, 'title'])
print('SBERT   :', top_similar(sbert_X, idx))
print('CodeBERT:', top_similar(codebert_X, idx))

sbert     shape=(3256, 384)  baris nol=0  norm rata-rata=1.000
codebert  shape=(3256, 768)  baris nol=0  norm rata-rata=18.070
Selisih maksimum (sendiri vs batch): 0.0

Soal acuan: Two Sum
SBERT   : [('Minimum Operations to Make Array Equal to Target', 0.904), ('Binary Search', 0.901), ('Create Target Array in the Given Order', 0.898), ('Two Sum II - Input Array Is Sorted', 0.897), ('Count Pairs Whose Sum is Less than Target', 0.891)]
CodeBERT: [('Partition Array into Disjoint Intervals', 0.996), ('Minimum Increment to Make Array Unique', 0.996), ('Add Two Numbers II', 0.996), ('The kth Factor of n', 0.996), ('Sum of Mutated Array Closest to Target', 0.996)]


## 7. Simpan

In [7]:
np.save(os.path.join(FEAT_DIR, 'sbert.npy'), sbert_X)
np.save(os.path.join(FEAT_DIR, 'codebert.npy'), codebert_X)

pd.DataFrame([stats_sbert, stats_cb]).to_csv(os.path.join(TAB_DIR, 'truncation_stats.csv'), index=False)

meta = {
    'n_rows': int(len(df)),
    'text_view': 'text_light (digits, symbols, case and stopwords preserved)',
    'fine_tuning': False,
    'sbert': {'model': SBERT_ID, 'max_seq_length': SBERT_MAXLEN, 'dim': int(sbert_X.shape[1]),
              'pooling': 'model default (CLS for bge-small), L2-normalized'},
    'codebert': {'model': CODEBERT_ID, 'max_seq_length': CODEBERT_MAXLEN, 'dim': int(codebert_X.shape[1]),
                 'pooling': 'mean of last hidden state with attention mask (not normalized)'},
    'truncation': {'sbert': stats_sbert, 'codebert': stats_cb},
}
with open(os.path.join(FEAT_DIR, 'transformer_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

assert np.load(os.path.join(FEAT_DIR, 'sbert.npy')).shape == sbert_X.shape
assert np.load(os.path.join(FEAT_DIR, 'codebert.npy')).shape == codebert_X.shape
print('Tersimpan di', FEAT_DIR)
pd.DataFrame([stats_sbert, stats_cb])

Tersimpan di /kaggle/working/features


,model,max_len,tokens_mean,tokens_median,tokens_p95,tokens_max,problems_truncated_n,problems_truncated_%,tokens_lost_%
0,BAAI/bge-small-en-v1.5,512,383.0,342.0,758.25,1935,633,19.44,9.58
1,microsoft/codebert-base,512,352.1,319.0,659.50,1572,464,14.25,6.09


## Ringkasan untuk paper (Bab III-D dan keterbatasan)

Salin dari output:
- model, dimensi, `max_seq_length`, dan jenis pooling (sudah ada di `transformer_meta.json`)
- persentase soal yang terpotong dan persentase token yang hilang (`truncation_stats.csv`). Tulis di keterbatasan: bagian yang terpotong adalah akhir deskripsi, tempat *constraints* biasanya berada.
- catatan: CodeBERT dilatih pada pasangan bahasa alami dan kode (bimodal), sedangkan masukan di sini hanya bahasa alami. Ini bisa membatasi keunggulannya.

Setelah selesai, unggah folder `features/` (bersama `glove.npy` dan `fasttext.npy` dari Notebook 03) sebagai satu Kaggle Dataset private untuk Notebook 05-06.